# RAG-based Semantic Post-Correction for ATC ASR

```
                        ↓
                        ↓
        ↓
Entity-specific RAG Correction
    ├── ① Instruction  → ICAO Doc 9432 Vector DB
    ├── ② Callsign     → OpenSky API + ICAO Airline Codes
    └── ③ Facility     → file_key Prior + Airport DB
        ↓
```

- `atco2_gold_clean.csv` — Gold annotation
- Anthropic API Key ([console.anthropic.com](https://console.anthropic.com))

---

In [1]:
!pip install faiss-cpu sentence-transformers jiwer transformers requests pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Step 1: Restart runtime first, then run only this cell
!pip install -q transformers torchaudio

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_ID  = 'Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc'
atc_processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
atc_model_asr = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
atc_model_asr.eval()
print('ATC ASR model loaded')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


/opt/jupyter/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 424/424 [00:00<00:00, 37824.30it/s]

ATC ASR model loaded


In [6]:
# Colab file upload removed. Using local paths in Jupyter.
import os

In [7]:
# (Colab files.upload removed — using local paths in Jupyter)
import os, glob

In [8]:
import os

# Place baseline_result.csv and atco2_gold_clean.csv
# in the same folder as this notebook, or set full paths below.
BASELINE_CSV = 'baseline_result.csv'
GOLD_CSV     = 'atco2_gold_clean.csv'

assert os.path.exists(BASELINE_CSV), f'File not found: {BASELINE_CSV}'
assert os.path.exists(GOLD_CSV),     f'File not found: {GOLD_CSV}'
print(f'Found: {BASELINE_CSV}')
print(f'Found: {GOLD_CSV}')

Found: baseline_result.csv
Found: atco2_gold_clean.csv


In [9]:
import pandas as pd

baseline_df = pd.read_csv(BASELINE_CSV)
gold_df     = pd.read_csv(GOLD_CSV)

baseline_df = baseline_df.dropna(subset=['gold', 'asr']).reset_index(drop=True)

print(f'Baseline rows : {len(baseline_df)}')
print(f'Avg WER       : {baseline_df["WER"].mean():.4f}')
print(f'Error type counts:\n{baseline_df["error_type"].value_counts()}')

Baseline rows : 547
Avg WER       : 0.5190
Error type counts:
error_type
facility name    6
phonetic code    6
callsign         5
recognition      2
QNH              1
instruction      1
Name: count, dtype: int64


In [12]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# [Korean print removed]

class FAISSVectorDB:
    """Accepts text list, FAISS  build top-k search   """

    def __init__(self, docs: list[str]):
        self.docs = docs
        vecs = embedder.encode(docs, show_progress_bar=False).astype('float32')
        faiss.normalize_L2(vecs)
        self.index = faiss.IndexFlatIP(vecs.shape[1])  # cosine similarity (L2-normalized inner product)
        self.index.add(vecs)

    def search(self, query: str, k: int = 5) -> list[str]:
        vec = embedder.encode([query]).astype('float32')
        faiss.normalize_L2(vec)
        scores, indices = self.index.search(vec, k)
        return [self.docs[i] for i in indices[0] if i < len(self.docs)]

if BACKEND == "groq":
    from groq import Groq as _Groq
    _groq_client = _Groq(api_key=os.environ.get("API_KEY")GROQ_API_KEY")")

    def call_claude(prompt: str, max_tokens: int = 300) -> str:
        resp = _groq_client.chat.completions.create(
            model=_GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.0,
        )
        return resp.choices[0].message.content.strip()

elif BACKEND == "ollama":
    import requests as _req

    def call_claude(prompt: str, max_tokens: int = 300) -> str:
        resp = _req.post(
            "http://localhost:11434/api/generate",
            json={"model": _OLLAMA_MODEL, "prompt": prompt,
                  "stream": False, "options": {"num_predict": max_tokens}},
            timeout=60,
        )
        return resp.json()["response"].strip()

elif BACKEND == "gemini":
    import google.generativeai as _genai
    _genai.configure(api_key=os.environ.get("API_KEY")GEMINI_API_KEY")")
    _gemini_model = _genai.GenerativeModel("gemini-2.5-flash")

    def call_claude(prompt: str, max_tokens: int = 300) -> str:
        resp = _gemini_model.generate_content(
            prompt,
            generation_config={"max_output_tokens": max_tokens, "temperature": 0.0},
        )
        return resp.text.strip()

else:
    raise ValueError(f"Unknown BACKEND: {BACKEND}")

print(f"LLM backend: {BACKEND}")

# [Korean print removed]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1567.98it/s]


LLM backend: groq


In [13]:
# -- STEP 2-B. ATC BERT NER Model Load --
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline as hf_pipeline
)
import torch

NER_MODEL_ID = 'Jzuluaga/bert-base-ner-atc-en-atco2-1h'
# [Korean print removed]

_ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_ID)
_ner_model     = AutoModelForTokenClassification.from_pretrained(NER_MODEL_ID)

ner_model = hf_pipeline(
    'token-classification',
    #model=_ner_model,
    #tokenizer=_ner_tokenizer,
    #aggregation_strategy='simple',
    model='./atc_ner_best',
    aggregation_strategy='first',
    device=-1   # CPU
)
print('NER model loaded\n')

_test = 'swiss two zero four kilo climb to flight level three four zero'
# [Korean print removed]
for e in ner_model(_test):
    print(f'  [{e["entity_group"]:10s}] {e["word"]:30s} score={e["score"]:.3f}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1233.91it/s]


NER model loaded

  [CALLSIGN  ] swiss two zero four kilo       score=0.999
  [COMMAND   ] climb to                       score=0.999
  [VALUE     ] flight level three four zero   score=0.996


In [15]:
# -- ICAO Doc 9432 Core Phraseology --
ICAO_PHRASES = [
    ('climb',              'climb to flight level [FL]',            'altitude instruction ascending aircraft'),
    ('descend',            'descend to flight level [FL]',           'altitude instruction descending aircraft'),
    ('cleared to land',    'cleared to land runway [RWY]',           'landing clearance given to aircraft'),
    ('line up and wait',   'line up runway [RWY] and wait',          'aircraft holds on runway before takeoff'),
    ('cleared for takeoff','cleared for takeoff runway [RWY]',       'takeoff clearance issued'),
    ('contact',            'contact [FACILITY] [FREQ]',              'frequency handoff to next sector'),
    ('squawk',             'squawk [CODE]',                          'transponder code assignment'),
    ('radar contact',      'radar contact',                          'controller confirms radar identification'),
    ('turn left heading',  'turn left heading [HDG]',                'directional instruction left'),
    ('turn right heading', 'turn right heading [HDG]',               'directional instruction right'),
    ('maintain',           'maintain flight level [FL]',             'hold current altitude'),
    ('report',             'report [WAYPOINT/POSITION]',             'position reporting instruction'),
    ('frequency change',   'frequency change approved',              'pilot approved to change frequency'),
    ('hold short',         'hold short of runway [RWY]',             'ground instruction stop before runway'),
    ('taxi via',           'taxi via [TAXIWAY] to [STAND/RWY]',      'ground movement instruction'),
    ('continue',           'continue approach',                      'approach continuation clearance'),
    ('go around',          'go around',                              'missed approach instruction'),
    ('speed',              'reduce speed to [KT] knots',             'speed restriction'),
    ('wind',               'wind [DIR] degrees [SPD] knots',         'wind information'),
    ('qnh',                'QNH [VALUE]',                            'altimeter setting instruction'),
    ('cross',              'cross runway [RWY]',                     'ground instruction cross runway'),
]

instruction_docs = [f"{cmd}: {phrase} | {ctx}" for cmd, phrase, ctx in ICAO_PHRASES]
instruction_db   = FAISSVectorDB(instruction_docs)

import editdistance
INSTRUCTION_VOCAB = [cmd for cmd, _, _ in ICAO_PHRASES]

def edit_distance_match(word: str, threshold: int = 2):
    """Return closest instruction for a single word. None if dist > threshold."""
    candidates = [(w, editdistance.eval(word.lower(), w.lower())) for w in INSTRUCTION_VOCAB]
    best, dist = min(candidates, key=lambda x: x[1])
    return best if dist <= threshold else None

def correct_instruction(asr_text: str) -> str:
    """
    """
    relevant = instruction_db.search(asr_text, k=4)
    retrieved = "\n".join(f"  - {r}" for r in relevant)

    lines = [
        "You are an Air Traffic Control (ATC) transcript corrector.",
        f"ASR output (may contain errors): \"{asr_text}\"",
        "",
        "Relevant ICAO standard phrases retrieved:",
        retrieved,
        "",
        "## Task",
        "1. Correct ATC command words that are phonetically confused",
        "   (e.g. 'come' -> 'climb', 'ready' -> 'radar', 'out of' -> 'cross')",
        "2. Convert numeric digits in heading/altitude/FL to spoken form:",
        "   - heading 270       -> heading TWO SEVEN ZERO",
        "   - climb 5500 feet   -> climb FIVE THOUSAND FIVE HUNDRED feet",
        "   - FL 180            -> flight level ONE EIGHT ZERO",
        "   - descend 3000 feet -> descend THREE THOUSAND feet",
        "3. CRITICAL - Do NOT convert these:",
        "   - wind direction: 'wind 040 degrees' (near 'knots') -> leave unchanged",
        "   - runway numbers: 'runway 25' -> leave unchanged",
        "   - QNH values: 'QNH 1023' -> leave unchanged",
        "   - frequencies: '123.255' -> leave unchanged",
        "4. Do NOT change callsigns or facility names",
        "5. Return ONLY the corrected transcript, no explanation",
        "",
        "Corrected transcript:",
    ]
    prompt = "\n".join(lines)
    return call_claude(prompt)

# Test
test_cases = [
    'come to flight level three four zero',
    'ready i contact frequency one two one decimal five',
    'squock one two three four',
]
print('-- Instruction correction test --')
for t in test_cases:
    result = correct_instruction(t)
    print(f'  IN : {t}')
    print(f'  OUT: {result}')
    print()

-- Instruction correction test --
  IN : come to flight level three four zero
  OUT: climb to flight level THREE FOUR ZERO

  IN : ready i contact frequency one two one decimal five
  OUT: "radar contact frequency one two one decimal five"

  IN : squock one two three four
  OUT: squawk ONE TWO THREE FOUR



In [16]:
import requests
import pandas as pd

def _scrape_wiki_airlines() -> dict:
    mapping = {}
    for letter in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
        url = f'https://en.wikipedia.org/wiki/List_of_airline_codes_({letter})'
        try:
            tables = pd.read_html(url, header=0)
            for tbl in tables:
                cols = [str(c).lower() for c in tbl.columns]
                icao_col  = next((i for i,c in enumerate(cols) if 'icao' in c), None)
                call_col  = next((i for i,c in enumerate(cols) if 'call' in c or 'telephony' in c), None)
                name_col  = next((i for i,c in enumerate(cols) if 'airline' in c or 'name' in c), None)
                if icao_col is None or call_col is None:
                    continue
                for _, row in tbl.iterrows():
                    icao3 = str(row.iloc[icao_col]).strip().upper()
                    tel   = str(row.iloc[call_col]).strip().upper()
                    name  = str(row.iloc[name_col]).strip() if name_col is not None else ''
                    if len(icao3) == 3 and icao3.isalpha() and tel not in ('NAN','','-','\\N'):
                        mapping[icao3] = (tel, name)
        except Exception:
            continue
    return mapping

_AIRLINE_FALLBACK = {
    'ANA': ('ALL NIPPON',   'All Nippon Airways'),
    'SWR': ('SWISS',        'Swiss International Air Lines'),
    'EZS': ('EASY',         'easyJet Switzerland'),
    'DLH': ('LUFTHANSA',    'Lufthansa'),
    'BAW': ('SPEEDBIRD',    'British Airways'),
    'AFR': ('AIR FRANCE',   'Air France'),
    'KLM': ('KLM',          'KLM Royal Dutch Airlines'),
    'JTE': ('JETSTAR',      'Jetstar Airways'),
    'QFA': ('QANTAS',       'Qantas Airways'),
    'VOZ': ('VELOCITY',     'Virgin Australia'),
    'AUA': ('AUSTRIAN',     'Austrian Airlines'),
    'CSN': ('SOUTHERN',     'China Southern Airlines'),
    'CCA': ('AIR CHINA',    'Air China'),
    'RXA': ('REX',          'Regional Express'),
    'SIA': ('SINGAPORE',    'Singapore Airlines'),
    'UAE': ('EMIRATES',     'Emirates'),
    'THY': ('TURKISH',      'Turkish Airlines'),
    'OAL': ('OLYMPIC',      'Olympic Air'),
    'RYR': ('RYANAIR',      'Ryanair'),
    'KAL': ('KOREAN AIR',   'Korean Air'),
    'AAR': ('ASIANA',       'Asiana Airlines'),
    'WZZ': ('WIZZAIR',      'Wizz Air'),
    'EWG': ('EUROWINGS',    'Eurowings'),
    'SAS': ('SCANDINAVIAN', 'Scandinavian Airlines'),
    'AAL': ('AMERICAN',     'American Airlines'),
    'UAL': ('UNITED',       'United Airlines'),
    'DAL': ('DELTA',        'Delta Air Lines'),
}

import os as _os

_CSV_PATHS = [
    'airlines.csv',
    '/content/airlines.csv',
    '/mnt/user-data/uploads/airlines.csv',
    _os.path.join(_os.getcwd(), 'airlines.csv'),
]

def _load_csv_airlines(path: str) -> dict:
    """airlines.csv -> {ICAO3: (telephony, name)}

    """
    mapping = {}
    try:
        raw = pd.read_csv(path, nrows=1)
        cols_lower = [str(c).strip().lower() for c in raw.columns]

        if any("lett" in c or "callsign" in c for c in cols_lower):
            df = pd.read_csv(path)
            col_icao = next(
                (c for c in df.columns if "lett" in c.lower() or c.lower() in ("icao","code3")),
                None)
            col_call = next(
                (c for c in df.columns if "callsign" in c.lower()),
                None)
            col_name = next(
                (c for c in df.columns if "airline" in c.lower() or c.lower() == "name"),
                None)
            if col_icao is None or col_call is None:
                return mapping
            for _, row in df.iterrows():
                icao3    = str(row[col_icao]).strip().upper()
                callsign = str(row[col_call]).strip().upper()
                name     = str(row[col_name]).strip() if col_name else ""
                if (len(icao3) == 3 and icao3.isalpha()
                        and callsign not in ("NAN", "NAT", "", "-", "NONE")):
                    mapping[icao3] = (callsign, name)

        else:
            df = pd.read_csv(
                path, header=None,
                names=["id","name","alias","iata","icao","callsign","country","active"]
            )
            for _, row in df.iterrows():
                icao3    = str(row["icao"]).strip().upper()
                callsign = str(row["callsign"]).strip().upper()
                name     = str(row["name"]).strip()
                if (len(icao3) == 3 and icao3.isalpha()
                        and callsign not in ("NAN", "", "-")):
                    mapping[icao3] = (callsign, name)

    except Exception as _csv_e:
    return mapping

_csv_airlines = {}
for _csv_path in _CSV_PATHS:
    if _os.path.exists(_csv_path):
        _csv_airlines = _load_csv_airlines(_csv_path)
        break

if _csv_airlines:
    AIRLINE_PHONETIC = {**_csv_airlines, **_AIRLINE_FALLBACK}
else:
    try:
        _wiki_airlines = _scrape_wiki_airlines()
        if len(_wiki_airlines) > 50:
            AIRLINE_PHONETIC = {**_wiki_airlines, **_AIRLINE_FALLBACK}
        else:
            AIRLINE_PHONETIC = _AIRLINE_FALLBACK
    except Exception as e:
        AIRLINE_PHONETIC = _AIRLINE_FALLBACK

print(f'  ANA: {AIRLINE_PHONETIC.get("ANA")}')
print(f'  JTE: {AIRLINE_PHONETIC.get("JTE")}')

NUM_WORD_TO_DIGIT = {
    'zero':'0','one':'1','two':'2','three':'3','four':'4',
    'five':'5','six':'6','seven':'7','eight':'8','nine':'9',
}
PHONETIC_ALPHA = {
    'alpha':'A','bravo':'B','charlie':'C','delta':'D','echo':'E',
    'foxtrot':'F','golf':'G','hotel':'H','india':'I','juliet':'J',
    'kilo':'K','lima':'L','mike':'M','november':'N','oscar':'O',
    'papa':'P','quebec':'Q','romeo':'R','sierra':'S','tango':'T',
    'uniform':'U','victor':'V','whiskey':'W','x-ray':'X','yankee':'Y','zulu':'Z',
}

def opensky_fetch_callsigns(icao_code: str) -> list:
    AIRPORT_COORDS = {
        'LSGS':(46.2196,7.3267),'LKPR':(50.1008,14.2600),
        'LSZB':(46.9141,7.4972),'LSZH':(47.4582,8.5555),
        'YSSY':(-33.9461,151.177),'LZIB':(48.1702,17.2127),
        'LKTB':(49.1513,16.6944),
    }
    if icao_code not in AIRPORT_COORDS:
        return []
    lat, lon = AIRPORT_COORDS[icao_code]
    try:
        resp = requests.get(
            'https://opensky-network.org/api/states/all',
            params={'lamin':lat-1.5,'lamax':lat+1.5,'lomin':lon-1.5,'lomax':lon+1.5},
            timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            return list({s[1].strip() for s in (data.get('states') or []) if s[1]})
    except Exception:
        pass
    return []

def icao_callsign_to_spoken(icao_cs: str) -> str:
    prefix = icao_cs[:3].upper()
    spoken_airline, _ = AIRLINE_PHONETIC.get(prefix, (prefix, prefix))
    spoken_parts = []
    for ch in icao_cs[3:]:
        if ch.isdigit():
            word = [k for k,v in NUM_WORD_TO_DIGIT.items() if v == ch]
            spoken_parts.append(word[0].upper() if word else ch)
        elif ch.isalpha():
            phonetic = [k.upper() for k,v in PHONETIC_ALPHA.items() if v == ch.upper()]
            spoken_parts.append(phonetic[0] if phonetic else ch)
    return f"{spoken_airline} {' '.join(spoken_parts)}"

def build_callsign_db(icao_code: str) -> FAISSVectorDB:
    live = opensky_fetch_callsigns(icao_code)
    docs = [f"{icao_callsign_to_spoken(cs)} | ICAO: {cs}" for cs in live[:50]]
    for icao3, (telephony, _) in AIRLINE_PHONETIC.items():
        for num in range(1, 1000, 100):
            docs.append(f"{icao_callsign_to_spoken(f'{icao3}{num:03d}')} | Airline: {telephony}")
    return FAISSVectorDB(docs or ['callsign database empty'])

CSV 로드 성공: airlines.csv -> 5176개 항공사
최종 AIRLINE_PHONETIC: 5177개 (CSV + fallback)
  ANA: ('ALL NIPPON', 'All Nippon Airways')
  JTE: ('JETSTAR', 'Jetstar Airways')


In [17]:
import re as _re_cs

def _word_overlap_score(span: str, db_entry: str) -> float:
    span_words = set(span.upper().split())
    db_spoken = db_entry.split('|')[0].strip().upper()
    db_words  = set(db_spoken.split())
    if not db_words:
        return 0.0
    return len(span_words & db_words) / len(db_words)

def correct_callsign(asr_text: str, callsign_db: FAISSVectorDB) -> str:
    """
    """
    relevant = callsign_db.search(asr_text, k=5)

    overlap_score = max(
        (_word_overlap_score(asr_text, r) for r in relevant),
        default=0.0
    )
    best_candidate = relevant[0] if relevant else ''

    prompt = f"""You are an ATC callsign corrector.

ASR output (may have callsign errors): \"{asr_text}\"

Candidate correct callsigns (spoken form | ICAO):
{chr(10).join(f'  - {r}' for r in relevant)}

Word overlap score between ASR and best candidate: {overlap_score:.2f}

## Rules (STRICT)
1. NEVER add words NOT present in the ASR text.
   - If ASR says "Charlie Papa", do NOT output "HOTEL CHARLIE PAPA".
   - The corrected callsign must only contain words audible in the ASR.
2. If overlap_score < 0.40, return the ASR callsign span unchanged.
   A low score means the retrieved candidate does not match this recording.
3. Callsign formats:
   - Registration (e.g., HB-OSC): phonetic letters → "HOTEL OSCAR SIERRA CHARLIE"
   - Airline flight (e.g., Swiss 204K): airline telephony + digits individually
     → "SWISS TWO ZERO FOUR KILO"  (NOT "SWISS 204K", NOT "SWISS TWO HUNDRED FOUR")
4. Digit rule: convert EACH digit individually.
   "3043" → "THREE ZERO FOUR THREE"  (NOT "THIRTY FORTY THREE")
5. Do NOT correct if the span is a communication word: ROGER, AFFIRM, WILCO, NEGATIVE.
6. Do NOT change instructions or facility names.
7. Return ONLY the corrected transcript, no explanation.

Corrected transcript:"""

    corrected = call_claude(prompt)

    if overlap_score < 0.40:

    return corrected

test_db = FAISSVectorDB([
    'JETSTAR NINE ONE TWO | ICAO: JTE912',
    'SWISS TWO ZERO FOUR KILO | ICAO: SWR204K',
    'ALL NIPPON EIGHT THREE NINER | ICAO: ANA839',
    'REX SIX EIGHT SIX EIGHT | ICAO: RXA6868',
])
test_cases = [
    'DIFTER912 contact departure',
    'CIS204K departure bye',
    'all nippon 839 climb FL180',
]
print('-- Callsign correction test --')
for t in test_cases:
    result = correct_callsign(t, test_db)
    print(f'  IN : {t}')
    print(f'  OUT: {result}')
    print()

-- Callsign correction test --
  IN : DIFTER912 contact departure
  OUT: DIFTER912 contact departure

  IN : CIS204K departure bye
  OUT: CIS204K departure bye

  IN : all nippon 839 climb FL180
  OUT: all nippon 839 climb FL180

Corrected callsign: ALL NIPPON EIGHT THREE NINER | ICAO: ANA839

Corrected transcript: "all NIPPON eight three niner climb FL180"



In [18]:
# -- ATC Facility Knowledge Base --
FACILITY_DB = {
    'LSGS': {
        'name': 'Sion',
        'country': 'Switzerland',
        'services': {
            'Tower':          'Sion Tower',
            'Ground':         'Sion Ground',
            'ApronN':         'Sion Apron',
            'ApronS':         'Sion Apron',
        },
    },
    'LKPR': {
        'name': 'Ruzyne',
        'country': 'Czech Republic',
        'services': {
            'Tower':   'Prague Tower',
            'Radar':   'Prague Radar',
            'Ground':  'Prague Ground',
        },
    },
    'LSZB': {
        'name': 'Bern',
        'country': 'Switzerland',
        'services': {
            'Tower':  'Bern Tower',
            'Ground': 'Bern Ground',
        },
    },
    'LSZH': {
        'name': 'Zurich',
        'country': 'Switzerland',
        'services': {
            'Tower':    'Zurich Tower',
            'Approach': 'Zurich Approach',
            'Radar':    'Zurich Radar',
            'Ground':   'Zurich Ground',
        },
    },
    'YSSY': {
        'name': 'Sydney',
        'country': 'Australia',
        'services': {
            'Tower':            'Sydney Tower',
            'Ground':           'Sydney Ground',
            'Approach':         'Sydney Approach',
            'Approach-Radar':   'Sydney Approach',
        },
    },
    'LZIB': {
        'name': 'Stefanik',
        'country': 'Slovakia',
        'services': {
            'Tower':  'Bratislava Tower',
            'Ground': 'Bratislava Ground',
        },
    },
    'LKTB': {
        'name': 'Brno',
        'country': 'Czech Republic',
        'services': {
            'Tower':   'Brno Tower',
            'Approach':'Brno Approach',
        },
    },
}

def parse_file_key(file_key: str) -> dict:
    """
    file_key: 'LSGS_SION_Ground_Control_121_7MHz_20210504_062720'
    """
    parts = file_key.split('_')
    return {
        'icao':    parts[0],           # 'LSGS'
        'airport': parts[1],           # 'SION'
        'service': parts[2],           # 'Ground'
        'freq':    f'{parts[3]}.{parts[4].replace("MHz","")}',
    }

def get_expected_facility(file_key: str) -> str:
    """file_key  facility callsign return (hard prior)"""
    meta = parse_file_key(file_key)
    info = FACILITY_DB.get(meta['icao'])
    if not info:
        return meta['airport'].capitalize() + ' ' + meta['service']
    services = info.get('services', {})
    return services.get(meta['service'], f"{info['name']} {meta['service']}")

def build_facility_vector_db() -> FAISSVectorDB:
    """facility nameAlso store phonetic variants  search  """
    docs = []
    for icao, info in FACILITY_DB.items():
        for svc, callsign in info['services'].items():
            docs.append(f"{callsign} | {icao} | {info['name']} | {info['country']}")
    return FAISSVectorDB(docs)

facility_db = build_facility_vector_db()

def correct_facility(asr_text: str, file_key: str) -> str:
    """
    """

    expected = get_expected_facility(file_key)
    meta     = parse_file_key(file_key)
    relevant = facility_db.search(asr_text, k=3)

    facility_kws = ['tower', 'ground', 'approach', 'radar', 'control',
                    'departure', 'arrival', 'centre', 'center']
    asr_words = asr_text.lower().split()
    has_facility_mention = any(
        _editdist.eval(w, kw) / len(kw) <= 0.40
        for w in asr_words
        for kw in facility_kws
    )
    if not has_facility_mention:

    cand_block = '\n'.join('  - ' + r for r in relevant)
    prompt = (
        'You are an ATC facility name corrector.\n'
        '\n'
        'Recording metadata:\n'
        '  Airport ICAO  : ' + meta['icao'] + '\n'
        '  Airport name  : ' + meta['airport'] + '\n'
        '  ATC service   : ' + meta['service'] + '\n'
        '  Frequency     : ' + meta['freq'] + ' MHz\n'
        '  CORRECT facility callsign: "' + expected + '"\n'
        '\n'
        'ASR output (may have facility name errors): "' + asr_text + '"\n'
        '\n'
        'Similar facility names from database:\n'
        + cand_block + '\n'
        '\n'
        '## Rules\n'
        '1. Replace only the wrongly-spoken facility name with "' + expected + '".\n'
        '2. Do NOT change callsigns, instructions, or numbers.\n'
        '3. If uncertain (confidence < 0.65), return the original unchanged.\n'
        '4. Return ONLY the corrected transcript, no explanation.\n'
        '\n'
        'Corrected transcript:'
    )
    return call_claude(prompt)
# [Korean print removed]
test_cases = [
    ('Sunita, test out 767. Test out 767, Sunita.',      'YSSY_SYDNEY_Tower_120_5MHz_20210501_100000'),
    ('Sydney Teagrae, QNH 529. QNH 529, Sydney Teagrae.','YSSY_SYDNEY_Tower_120_5MHz_20210501_100000'),
]
for asr, fkey in test_cases:
    result = correct_facility(asr, fkey)
    print(f'  IN : {asr}')
    print(f'  OUT: {result}')
    print()

  IN : Sunita, test out 767. Test out 767, Sunita.
  OUT: Sunita, test out 767. Test out 767, Sunita.

  IN : Sydney Teagrae, QNH 529. QNH 529, Sydney Teagrae.
  OUT: Sydney Teagrae, QNH 529. QNH 529, Sydney Teagrae.



In [19]:
import re
from functools import lru_cache

class ATCRAGPipeline:
    """
    error_type   entity RAG   .
    error_type      (general correction).
    """

    def __init__(self):
        self._callsign_db_cache: dict[str, FAISSVectorDB] = {}  # ICAOper-slot caching
        print('ATCRAGPipeline initialized')

    def _get_callsign_db(self, icao_code: str) -> FAISSVectorDB:
        if icao_code not in self._callsign_db_cache:
            # [Korean print removed]
            self._callsign_db_cache[icao_code] = build_callsign_db(icao_code)
        return self._callsign_db_cache[icao_code]

    def correct(self, asr_text: str, file_key: str, error_type: str | None = None) -> str:
        """
        error_type   RAG  .
        error_type None  NaN general LLM  .
        """
        import math
        if not isinstance(error_type, str) or (isinstance(error_type, float) and math.isnan(error_type)):
            error_type = None

        if error_type == 'instruction':
            return correct_instruction(asr_text)

        elif error_type == 'callsign':
            icao = parse_file_key(file_key)['icao']
            db   = self._get_callsign_db(icao)
            return correct_callsign(asr_text, db)

        elif error_type in ('facility name', 'facility'):
            return correct_facility(asr_text, file_key)

        elif error_type == 'phonetic code':
            step1 = correct_instruction(asr_text)
            icao  = parse_file_key(file_key)['icao']
            db    = self._get_callsign_db(icao)
            return correct_callsign(step1, db)

        elif error_type == 'recognition':
            return asr_text

        else:
            return self._general_correction(asr_text, file_key)

    def _general_correction(self, asr_text: str, file_key: str) -> str:
        expected_facility = get_expected_facility(file_key)
        meta = parse_file_key(file_key)

        prompt = f"""You are an ATC (Air Traffic Control) transcript corrector.

Recording context:
  - Airport: {meta['airport']} ({meta['icao']}), {FACILITY_DB.get(meta['icao'], {}).get('country', 'unknown')}
  - ATC unit: {expected_facility}
  - Frequency: {meta['freq']} MHz

ASR transcript to correct: \"{asr_text}\"

## Correction rules
1. Fix phonetically confused command words (climb/come, cleared/ready, cross/out of, etc.)
2. Convert numeric heading/altitude/FL to spoken form:
   - heading 270 → heading TWO SEVEN ZERO
   - FL 180 → flight level ONE EIGHT ZERO
   - climb 3000 feet → climb THREE THOUSAND feet
   EXCEPTION — do NOT convert: wind direction (near 'knots'), runway numbers, QNH, frequencies
3. Fix callsign: use airline telephony name + individual digits
   NEVER add prefix words not spoken in the ASR (e.g. do not add 'HOTEL' if not said)
4. Replace wrong facility names with: \"{expected_facility}\"
   Only if facility name is explicitly mentioned in ASR.

## Speaker Type Detection
- If ASR contains (we are, we have, not ready, request, negative, wilco, roger,
  affirm, standing by) → PILOT speech. Do NOT convert to ATC commands.

## STRICT constraints
- Do NOT add any word not present in the original ASR output.
- Do NOT repeat any phrase already in the transcript.
- If unsure whether a word is wrong, keep it as-is.

Return ONLY the corrected transcript, no explanation:"""

        return call_claude(prompt)

pipeline = ATCRAGPipeline()
# [Korean print removed]

ATCRAGPipeline initialized


In [20]:
# -- 5-1. WAV Input -> wav2vec2 ATC ASR --
import os, glob, torch, numpy as np
import soundfile as sf
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# pip install soundfile (if not installed)
# !pip install soundfile

# -- Load ATC-specific ASR model --
MODEL_ID  = 'Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc'
print(f'Loading ATC ASR model: {MODEL_ID}')
_asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
_asr_model     = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
_asr_model.eval()
print('ATC ASR model loaded\n')

def _resample(audio: np.ndarray, orig_sr: int, target_sr: int = 16000) -> np.ndarray:
    """Simple linear resample using numpy (no torchaudio/ffmpeg needed)"""
    if orig_sr == target_sr:
        return audio
    duration  = len(audio) / orig_sr
    new_len   = int(duration * target_sr)
    old_idx   = np.linspace(0, len(audio) - 1, new_len)
    return np.interp(old_idx, np.arange(len(audio)), audio)

def transcribe_wav(wav_path: str) -> str:
    """soundfile + wav2vec2: no ffmpeg, no torchaudio needed"""
    # Load audio with soundfile
    audio, sr = sf.read(wav_path, dtype='float32', always_2d=False)

    # Stereo → mono
    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    # Resample to 16kHz
    if sr != 16000:
        audio = _resample(audio, sr, 16000)

    inputs = _asr_processor(
        audio,
        sampling_rate=16000,
        return_tensors='pt',
        padding=True
    )
    with torch.no_grad():
        logits = _asr_model(**inputs).logits

    ids = torch.argmax(logits, dim=-1)
    return _asr_processor.batch_decode(ids)[0].lower()

# -- Scan WAV files --
WAV_DIR    = '1hour_train/test/'   # change if WAV files are elsewhere
_wav_files = sorted(
    glob.glob(os.path.join(WAV_DIR, '*.wav')) +
    glob.glob(os.path.join(WAV_DIR, '*.WAV'))
)

if not _wav_files:
    print(f'[WARN] No WAV files found in: {os.path.abspath(WAV_DIR)}')
    print('       Set WAV_DIR to the folder containing your WAV files.')
else:
    print(f'Found {len(_wav_files)} WAV file(s)')

wav_asr_results = {}

for wav_fullpath in _wav_files:
    fname        = os.path.basename(wav_fullpath)
    file_key_wav = fname.replace('.wav', '').replace('.WAV', '')
    print(f'Processing: {fname}')
    try:
        asr_text = transcribe_wav(wav_fullpath)
        wav_asr_results[file_key_wav] = asr_text
        print(f'  ASR: {asr_text[:80]}')
    except Exception as _e:
        print(f'  [ERR] {type(_e).__name__}: {_e}')
    print()

print(f'ASR done: {len(wav_asr_results)} file(s)')

if wav_asr_results:
    file_key_wav = list(wav_asr_results.keys())[-1]
    asr_from_wav = wav_asr_results[file_key_wav]
else:
    file_key_wav = None
    asr_from_wav = None

# -- ASR Confidence Score extraction --
def get_asr_confidence(wav_path: str) -> float:
    """
    Extract CTC log-probability confidence from wav2vec2.
    Returns value in [0, 1]. Higher = model more certain.
    Used as reference-free ASR quality signal (no gold needed).
    """
    try:
        audio, sr = sf.read(wav_path, dtype='float32', always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != 16000:
            audio = _resample(audio, sr, 16000)
        inputs = _asr_processor(
            audio, sampling_rate=16000, return_tensors='pt', padding=True
        )
        with torch.no_grad():
            logits = _asr_model(**inputs).logits
        log_probs  = torch.nn.functional.log_softmax(logits, dim=-1)
        confidence = log_probs[0].max(dim=-1).values.mean().exp().item()
        return round(float(confidence), 4)
    except Exception as _e:
        print(f'  [WARN] confidence error: {_e}')
        return 0.5  # neutral fallback

# -- Store confidence alongside ASR text --
wav_confidences = {}  # {file_key: confidence}

for wav_fullpath in _wav_files:
    fname        = os.path.basename(wav_fullpath)
    file_key_wav = fname.replace('.wav', '').replace('.WAV', '')
    if file_key_wav in wav_asr_results:
        conf = get_asr_confidence(wav_fullpath)
        wav_confidences[file_key_wav] = conf
        print(f'Confidence [{file_key_wav[-30:]}]: {conf}')

Loading ATC ASR model: Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 46507.10it/s]


ATC ASR model loaded

Found 58 WAV file(s)
Processing: YSSY_SYDNEY_Tower_120_5MHz_20210501_200237.wav
  ASR: tolay waiting on release from

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_004327.wav
  ASR: esix contac dpartu contact departurein three two six

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_012007.wav
  ASR: runway three four left cleared to land and request bore thro two bravo fourf ava

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_012016.wav
  ASR: four zero nine approved a three to bravo  o troe bravo  for thank you jsta four 

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_012942.wav
  ASR: ratar distar seven six three three seven six tr three

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_024039.wav
  ASR: rek sixty et sixty eight contact departure rek sixty eight six ty eight

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_034700.wav
  ASR: gl nine fourone nine fourgod

Processing: YSSY_SYDNEY_Tower_120_5MHz_20210502_040816.wav
  ASR: edincon six fourty te 

---
## STEP 5-1b. Reference-Free ASR Quality Assessment

|-------|-------------|----------|

In [21]:
import pandas as pd, os

if os.path.exists(ASR_CSV):
    asr_df = pd.read_csv(ASR_CSV)
    
    wav_asr_results = dict(zip(asr_df['file_key'], asr_df['asr_text'].fillna('')))
    
    wav_confidences = dict(zip(asr_df['file_key'], asr_df['confidence'].fillna(0.5)))
    
else:

✅ asr_results_uwb-atcc_lmTrue_last.csv 로드 완료: 58개
   샘플: ('YSSY_SYDNEY_Tower_120_5MHz_20210501_200237', 'toy waiting on release from')


In [22]:
import re as _reqa

if 'NATO_PHONETIC' not in dir():
    NATO_PHONETIC = {
        'alpha','bravo','charlie','delta','echo','foxtrot','golf',
        'hotel','india','juliet','kilo','lima','mike','november',
        'oscar','papa','quebec','romeo','sierra','tango','uniform',
        'victor','whiskey','xray','yankee','zulu',
        'zero','one','two','three','four','five',
        'six','seven','eight','nine',
    }

if 'INSTRUCTION_VOCAB' not in dir():
    INSTRUCTION_VOCAB = [
        'climb','descend','cleared','contact','squawk','maintain',
        'turn','heading','hold','taxi','report','frequency',
        'radar','approach','departure','continue','go around',
        'line up','cross','wind','qnh','speed',
    ]

if 'AIRLINE_PHONETIC' not in dir():
    AIRLINE_PHONETIC = {
    'SWR': ('Swiss', 'Swiss International Air Lines'),
    'EZS': ('Easy', 'easyJet Switzerland'),
    'DLH': ('Lufthansa', 'Lufthansa'),
    'BAW': ('Speedbird', 'British Airways'),
    'AFR': ('Air France', 'Air France'),
    'KLM': ('KLM', 'KLM Royal Dutch Airlines'),
    'JTE': ('Jetstar', 'Jetstar Airways'),
    'QFA': ('Qantas', 'Qantas Airways'),
    'VOZ': ('Velocity', 'Virgin Australia'),
    'TYR': ('Tyrolean', 'Austrian Airlines'),
    'AUA': ('Austrian', 'Austrian Airlines'),
    'CSN': ('Southern', 'China Southern Airlines'),
    'CCA': ('Air China', 'Air China'),
    'RXA': ('Rex', 'Regional Express'),
    'SIA': ('Singapore', 'Singapore Airlines'),
    'UAE': ('Emirates', 'Emirates'),
    'THY': ('Turkish', 'Turkish Airlines'),
    'VKG': ('Viking', 'Viking Airlines'),
    'OAL': ('Olympic', 'Olympic Air'),
    'NSZ': ('Air Nostrum', 'Air Nostrum'),
    'IBB': ('Binter', 'Binter Canarias'),
    'BEE': ('Bair', 'Flybe'),
    'SLS': ('Silesia', 'Enter Air'),
    'WIF': ('Bair', 'Bair Airlines'),
    'ANA': ('Allnippon', 'All Nippon Airways')
    }

In [23]:
if 'is_phonetic_sequence' not in dir():
    def is_phonetic_sequence(span: str, threshold: float = 0.5) -> bool:
        """Return True if >= threshold fraction of words are NATO phonetic."""
        words = span.lower().split()
        if not words:
            return False
        phonetic_cnt = sum(1 for w in words if w in NATO_PHONETIC)
        return phonetic_cnt / len(words) >= threshold

In [24]:
import re as _reqa

def assess_asr_quality(asr_text: str, confidence: float) -> dict:
    """
    Reference-free ASR quality assessment based on ICAO standards.
    Used when no gold annotation is available (new test data).

    Criteria:
      1. ASR model confidence (CTC log-prob)
      2. ICAO vocabulary coverage
      3. ATC entity format validity (runway, FL, freq, callsign)
    """
    tokens   = asr_text.lower().split()
    atc_vocab = set(INSTRUCTION_VOCAB) | NATO_PHONETIC | {
        'flight', 'level', 'runway', 'heading', 'knots', 'degrees',
        'squawk', 'frequency', 'contact', 'tower', 'ground', 'approach',
        'departure', 'arrival', 'radar', 'wind', 'qnh', 'cleared',
        'roger', 'wilco', 'affirm', 'negative', 'standby', 'correction',
    }
    coverage = (sum(1 for t in tokens if t in atc_vocab)
                / max(len(tokens), 1))

    # Entity format validity checks
    has_runway    = bool(_reqa.search(r'runway\s+\d+[lrcLRC]?', asr_text, _reqa.I))
    has_fl        = bool(_reqa.search(r'flight\s+level\s+\d+', asr_text, _reqa.I))
    has_freq      = bool(_reqa.search(r'\d{3}\.\d{1,3}', asr_text))
    has_callsign  = any(w in asr_text.lower()
                        for w in [v[0].lower() for v in AIRLINE_PHONETIC.values()])
    has_phonetic  = is_phonetic_sequence(asr_text, threshold=0.3)

    entity_score = sum([has_runway, has_fl, has_freq,
                        has_callsign, has_phonetic]) / 5

    overall = round(confidence * 0.35 + coverage * 0.40 + entity_score * 0.25, 3)

    if overall > 0.75:
        level = 'SKIP'
    elif overall > 0.50:
        level = 'LIGHT'
    elif overall > 0.25:
        level = 'AGGRESSIVE'
    else:
        level = 'REJECT'

    return {
        'confidence':     confidence,
        'vocab_coverage': round(coverage, 3),
        'entity_score':   round(entity_score, 3),
        'overall':        overall,
        'correction_level': level,
        'details': {
            'runway': has_runway, 'flight_level': has_fl,
            'frequency': has_freq, 'callsign': has_callsign,
            'phonetic': has_phonetic,
        }
    }

# -- Run quality assessment on all WAV results --
wav_quality = {}  # {file_key: quality_dict}

print(f"{'File':38s} | {'Conf':>6} | {'Vocab':>6} | {'Entity':>6} | {'Overall':>7} | Level")
print('-' * 80)

for fkey, asr_txt in wav_asr_results.items():
    conf    = wav_confidences.get(fkey, 0.5)
    quality = assess_asr_quality(asr_txt, conf)
    wav_quality[fkey] = quality
    print(f"{fkey[-38:]:38s} | {conf:6.3f} | "
          f"{quality['vocab_coverage']:6.3f} | "
          f"{quality['entity_score']:6.3f} | "
          f"{quality['overall']:7.3f} | {quality['correction_level']}")

File                                   |   Conf |  Vocab | Entity | Overall | Level
--------------------------------------------------------------------------------
_SYDNEY_Tower_120_5MHz_20210501_200237 |  0.931 |  0.000 |  0.000 |   0.326 | AGGRESSIVE
_SYDNEY_Tower_120_5MHz_20210502_004327 |  0.925 |  0.778 |  0.200 |   0.685 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_012007 |  0.933 |  0.579 |  0.400 |   0.658 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_012016 |  0.871 |  0.562 |  0.400 |   0.630 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_012942 |  0.908 |  0.889 |  0.200 |   0.723 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_024039 |  0.954 |  0.545 |  0.200 |   0.602 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_034700 |  0.890 |  0.714 |  0.200 |   0.647 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_040816 |  0.842 |  0.667 |  0.200 |   0.611 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_041017 |  0.846 |  0.625 |  0.200 |   0.596 | LIGHT
_SYDNEY_Tower_120_5MHz_20210502_041102 |  0.903 |  0.500 |  0.400 |   0.61

In [25]:
import editdistance, re as _re

# STEP 5-2.  Slot Filling + Levenshtein Re-ranking

_fac_cands = []
for _icao_k, _fac_v in FACILITY_DB.items():
    for _svc, _label in _fac_v['services'].items():
        _fac_cands.append(_label)
_fac_cands = list(set(_fac_cands))          # deduplication
facility_slot_db = FAISSVectorDB(_fac_cands)
# [Korean print removed]

# ② CALLSIGN candidates  (AIRLINE_PHONETIC spoken form)
_cs_cands = []
for _icao_k, (_spk, _name) in AIRLINE_PHONETIC.items():
    _cs_cands.append(f'{_spk} | ICAO:{_icao_k}')
callsign_slot_db = FAISSVectorDB(_cs_cands)
# [Korean print removed]

_cmd_cands = [phrase[0] for phrase in ICAO_PHRASES]  # (command, full_phrase, ctx)
instruction_slot_db = FAISSVectorDB(_cmd_cands)
# [Korean print removed]

# ④ RUNWAY candidates  (01–36 + L/R/C)
_rwy_cands = [
    f'runway {n:02d}{s}'
    for n in range(1, 37)
    for s in ['', 'L', 'R', 'C']
]
runway_slot_db = FAISSVectorDB(_rwy_cands)
# [Korean print removed]

# ⑤ GREETING candidates
GREETING_CANDS = [
    'good morning', 'good afternoon', 'good evening', 'good day', 'hello'
]

print()

# ── Levenshtein Re-ranker ─────────────────────────────────────────────
def _lev_rerank(span: str, faiss_db, k: int = 10,
                threshold: float = 0.7) -> str:
    """
    span → faiss_db top-k search → Levenshtein return closest candidate.
     span  threshold   .
    """
    if not span or len(span.strip()) == 0:
        return span
    candidates = faiss_db.search(span, k=k)
    best, best_dist = span, float('inf')
    for cand in candidates:
        cand_spoken = cand.split('|')[0].strip()
        d = editdistance.eval(span.lower(), cand_spoken.lower())
        if d < best_dist:
            best_dist, best = d, cand_spoken
    ref_len = max(len(span), len(best), 1)
    if best_dist / ref_len < threshold:
        return best
    return span

# ── Step 1: ASR → Slot Mapping ───────────────────────────────────────
def map_to_slots(asr_text: str) -> dict:
    """
    BERT NER +  fallback ASR span  .
    return: {FACILITY, CALLSIGN, INSTRUCTION, RUNWAY, GREETING, EXTRA}
    """
    slots = {
        'FACILITY':    None,
        'CALLSIGN':    None,
        'INSTRUCTION': None,
        'RUNWAY':      None,
        'QNH':         None,
        'WIND':        None,
        'FREQUENCY':   None,
        'GREETING':    None,
        'EXTRA':       [],
    }

    try:
        entities = ner_model(asr_text)
    except Exception as _e:
        # [Korean print removed]
        entities = []

    print(f'  NER raw: {[(e["entity_group"], e["word"], round(e["score"],2)) for e in entities]}')

    for e in entities:
        grp  = e['entity_group'].upper()
        word = e['word'].strip('"\' ')
        sc   = e['score']
        if sc < 0.4:
            continue

        if 'CALL' in grp and slots['CALLSIGN'] is None:
            slots['CALLSIGN'] = word
        elif 'COMMAND' in grp and slots['INSTRUCTION'] is None:
            slots['INSTRUCTION'] = word
        elif 'VALUE' in grp:
            w_lower = word.lower()
            if _re.search(r'runway\s+', asr_text.lower()) and _re.search(r'\b\d+\s*[lrcLRC]?\b', word):
                if slots['RUNWAY'] is None:
                    slots['RUNWAY'] = word
            elif 'qnh' in asr_text.lower() and _re.search(
                    r'\b(zero|one|two|three|four|five|six|seven|eight|nine)[\s\w]*', w_lower):
                if slots['QNH'] is None:
                    slots['QNH'] = word
            elif _re.search(r'\b(degrees|knots|calm)\b', w_lower):
                if slots['WIND'] is None:
                    slots['WIND'] = word
            elif _re.search(r'\b\d+\s*decimal\s*\d+\b', w_lower):
                if slots['FREQUENCY'] is None:
                    slots['FREQUENCY'] = word
            else:
                slots['EXTRA'].append(word)

    txt_lower = asr_text.lower()

    # GREETING
    if slots['GREETING'] is None:
        _gm = _re.search(r'good\s+(?:morning|afternoon|evening|day)', txt_lower)
        if _gm:
            slots['GREETING'] = _gm.group()

    # RUNWAY
    if slots['RUNWAY'] is None:
        _rm = _re.search(r'runway\s+\d+\s*[lrcLRC]?', txt_lower)
        if _rm:
            slots['RUNWAY'] = _rm.group()

    if slots['INSTRUCTION'] is None:
        _icao_kws = [p[0] for p in ICAO_PHRASES]
        for _kw in _icao_kws:
            if _kw in txt_lower:
                slots['INSTRUCTION'] = _kw
                break

    if slots['FACILITY'] is None:
        _numeric_re = _re.compile(
            r'^(zero|one|two|three|four|five|six|seven|eight|nine|\d)', _re.I)
        non_numeric_extra = [
            x for x in slots['EXTRA']
            if not _numeric_re.match(x.strip())
        ]
        if non_numeric_extra:
            slots['FACILITY'] = non_numeric_extra[0]

    return slots

def rerank_slots(slots: dict) -> dict:
    corrected = dict(slots)

    if slots['FACILITY']:
        corrected['FACILITY'] = _lev_rerank(
            slots['FACILITY'], facility_slot_db, k=10, threshold=0.65)

    if slots['CALLSIGN']:
        corrected['CALLSIGN'] = _lev_rerank(
            slots['CALLSIGN'], callsign_slot_db, k=10, threshold=0.65)

    if slots['INSTRUCTION']:
        corrected['INSTRUCTION'] = _lev_rerank(
            slots['INSTRUCTION'], instruction_slot_db, k=5, threshold=0.6)

    if slots['RUNWAY']:
        corrected['RUNWAY'] = _lev_rerank(
            slots['RUNWAY'], runway_slot_db, k=5, threshold=0.6)

    if slots['GREETING']:
        best, bd = slots['GREETING'], float('inf')
        for g in GREETING_CANDS:
            d = editdistance.eval(slots['GREETING'].lower(), g)
            if d < bd:
                bd, best = d, g
        corrected['GREETING'] = best if bd < len(slots['GREETING']) * 0.6 else slots['GREETING']

    return corrected

def reconstruct(slots: dict) -> str:
    """
      ATC  .
    None  . : FACILITY > GREETING > CALLSIGN > INSTRUCTION > RUNWAY > EXTRA
    """
    order = ['FACILITY', 'GREETING', 'CALLSIGN', 'INSTRUCTION',
             'QNH', 'WIND', 'RUNWAY', 'FREQUENCY']
    parts = [slots[k] for k in order if slots.get(k)]
    parts.extend(slots.get('EXTRA', []))
    return ' '.join(parts)

def slot_filling_correct(asr_text: str, file_key: str = '') -> dict:
    """
    Slot Filling + Re-ranking  .
    file_key ATC  facility prior  .
    """
    # Slot mapping
    slots_raw = map_to_slots(asr_text)
    print(f'  Slots raw      : { {k:v for k,v in slots_raw.items() if k != "EXTRA"} }')

    if file_key and 'MHz' in file_key and file_key.count('_') >= 3:
        _meta = parse_file_key(file_key)
        _fac_prior = FACILITY_DB.get(_meta['icao'], {}).get(
            'services', {}).get(_meta['service'])
        _fac_kws = ["tower","ground","approach","radar","control",
                    "departure","arrival","centre","center"]
        if _fac_prior and any(
                editdistance.eval(w, kw) / len(kw) <= 0.40
                for w in asr_text.lower().split()
                for kw in _fac_kws):
            slots_raw['FACILITY'] = _fac_prior   # hard prior 

    # Re-ranking
    slots_corr = rerank_slots(slots_raw)
    print(f'  Slots corrected: { {k:v for k,v in slots_corr.items() if k != "EXTRA"} }')

    _all_empty = all(
        not slots_corr.get(k)
        for k in ['FACILITY','CALLSIGN','INSTRUCTION','RUNWAY',
                  'QNH','WIND','FREQUENCY','GREETING']
    ) and not slots_corr.get('EXTRA')
    rag_output = asr_text if _all_empty else reconstruct(slots_corr)

    log = []
    for k in ['FACILITY','CALLSIGN','INSTRUCTION','RUNWAY','GREETING']:
        raw = slots_raw.get(k)
        cor = slots_corr.get(k)
        if raw and cor and raw.lower() != cor.lower():
            log.append({'slot': k, 'before': raw, 'after': cor})

    return {
        'asr_output':   asr_text,
        'slots_raw':    slots_raw,
        'slots_corr':   slots_corr,
        'rag_output':   rag_output,
        'corrections':  log,
    }

wav_all_results = {}

if not wav_asr_results:
    print("[WARN] wav_asr_results is empty. Run cell 5-1 first.")
else:
    for fkey, asr_txt in wav_asr_results.items():
        print(f'\n{"="*60}')
        print(f'File: {fkey}')
        print(f'ASR : {asr_txt[:90]}')

        try:
            result = slot_filling_correct(asr_txt, fkey)
            wav_all_results[fkey] = result
            print(f'RAG : {result["rag_output"][:90]}')
            if result['corrections']:
                for c in result['corrections']:
                    print(f'  [{c["slot"]}] {c["before"]} → {c["after"]}')
            else:
                print('  (no correction)')
        except Exception as _ex:
            import traceback
            print(f'  [ERR] {type(_ex).__name__}: {_ex}')
            traceback.print_exc()
            wav_all_results[fkey] = {
                'asr_output': asr_txt, 'rag_output': asr_txt,
                'slots_raw': {}, 'slots_corr': {}, 'corrections': []
            }

    print(f'\nDone: {len(wav_all_results)} file(s)')

    wav_llm_results = wav_llm_results if 'wav_llm_results' in dir() else {}
    if wav_all_results and wav_llm_results:
        print(f'\n{"─"*100}')
        print(f'{"File":28s} | {"ASR":28s} | {"Levenshtein":28s} | {"LLM v3":28s}')
        print('─' * 100)
        for fk in wav_llm_results:
            asr_s = wav_asr_results.get(fk, '')[:26]
            lev_s = wav_all_results.get(fk, {}).get('rag_output', '')[:26]
            llm_s = wav_llm_results[fk].get('rag_output', '')[:26]
            print(f'{fk[-28:]:28s} | {asr_s:28s} | {lev_s:28s} | {llm_s:28s}')



File: YSSY_SYDNEY_Tower_120_5MHz_20210501_200237
ASR : toy waiting on release from
  NER raw: [('NAMED', 'toy', 0.99), ('COMMAND', 'waiting on release from', 0.98)]
  Slots raw      : {'FACILITY': None, 'CALLSIGN': None, 'INSTRUCTION': 'waiting on release from', 'RUNWAY': None, 'QNH': None, 'WIND': None, 'FREQUENCY': None, 'GREETING': None}
  Slots corrected: {'FACILITY': None, 'CALLSIGN': None, 'INSTRUCTION': 'waiting on release from', 'RUNWAY': None, 'QNH': None, 'WIND': None, 'FREQUENCY': None, 'GREETING': None}
RAG : waiting on release from
  (no correction)

File: YSSY_SYDNEY_Tower_120_5MHz_20210502_004327
ASR : six contact part contact departure in three two six
  NER raw: [('VALUE', 'six', 0.98), ('COMMAND', 'contact part', 0.87), ('COMMAND', 'contact', 0.92), ('COMMAND', 'departure', 0.47), ('VALUE', 'in three two six', 0.89)]
  Slots raw      : {'FACILITY': 'in three two six', 'CALLSIGN': None, 'INSTRUCTION': 'contact part', 'RUNWAY': None, 'QNH': None, 'WIND': None, 'FREQUE

In [26]:
import json as _json

# STEP 5-3.  Slot Filling + LLM Re-ranking  (v3)

COMMAND_STRUCTURE_MAP = {}
for _phrase_tuple in ICAO_PHRASES:
    _cmd, _full, _ctx = _phrase_tuple
    if any(w in _ctx for w in ['pilot', 'report', 'request', 'inbound']):
        _stype  = 'pilot'
        _struct = '[FACILITY] [GREETING] [CALLSIGN] [INSTRUCTION] [VALUE/RUNWAY]'
    else:
        _stype  = 'controller'
        _struct = '[CALLSIGN] [VALUE] [RUNWAY] [INSTRUCTION]'
    COMMAND_STRUCTURE_MAP[_cmd] = {
        'type': _stype, 'structure': _struct, 'example': _full,
    }
# [Korean print removed]

def detect_comm_structure(asr_text: str) -> dict:
    top_cmds = instruction_slot_db.search(asr_text, k=3)
    best_cmd, best_info, best_dist = None, None, float('inf')
    for cmd_cand in top_cmds:
        cmd_word = cmd_cand.strip()
        if cmd_word in COMMAND_STRUCTURE_MAP:
            d = editdistance.eval(asr_text.lower(), cmd_word.lower())
            if d < best_dist:
                best_dist, best_cmd, best_info = d, cmd_word, COMMAND_STRUCTURE_MAP[cmd_word]
    if best_info:
        return {'detected_cmd': best_cmd, 'comm_type': best_info['type'],
                'structure': best_info['structure'], 'example': best_info['example']}
    return {'detected_cmd': None, 'comm_type': 'unknown',
            'structure': '[FACILITY/CALLSIGN] [INSTRUCTION] [VALUE] [RUNWAY]', 'example': ''}

def _collect_candidates_v3(slots_raw: dict) -> dict:
    cands = {}

    if slots_raw.get('FACILITY'):
        fac_span  = slots_raw['FACILITY']
        fac_cands = facility_slot_db.search(fac_span, k=5)
        _best_dist = min(
            (editdistance.eval(fac_span.lower(), c.split('|')[0].strip().lower())
             for c in fac_cands),
            default=float('inf')
        )
        _threshold = len(fac_span) * 0.8
        if _best_dist < _threshold:
            cands['FACILITY'] = fac_cands
        else:
            pass  # no facility candidates

    if slots_raw.get('CALLSIGN'):
        if should_correct(slots_raw['CALLSIGN'], callsign_slot_db,
                          keep_thr=0.25, check_phonetic=True):
            cands['CALLSIGN'] = callsign_slot_db.search(slots_raw['CALLSIGN'], k=5)
        else:
            pass  # → callsign 
    if slots_raw.get('INSTRUCTION'):
        if should_correct(slots_raw['INSTRUCTION'], instruction_slot_db, keep_thr=0.2):
            cands['INSTRUCTION'] = instruction_slot_db.search(slots_raw['INSTRUCTION'], k=5)
        else:
            pass  # Already accurate instruction → 
    if slots_raw.get('RUNWAY'):
        if should_correct(slots_raw['RUNWAY'], runway_slot_db, keep_thr=0.2):
            cands['RUNWAY'] = runway_slot_db.search(slots_raw['RUNWAY'], k=5)
        else:
            pass  # Already accurate runway → 
    if slots_raw.get('GREETING'):
        cands['GREETING'] = GREETING_CANDS
    for _slot in ('QNH', 'WIND', 'FREQUENCY'):
        if slots_raw.get(_slot):
            cands[_slot] = [slots_raw[_slot]]
    return cands

NATO_PHONETIC = {
    'alpha','bravo','charlie','delta','echo','foxtrot','golf',
    'hotel','india','juliet','kilo','lima','mike','november',
    'oscar','papa','quebec','romeo','sierra','tango','uniform',
    'victor','whiskey','xray','yankee','zulu',
    'zero','one','two','three','four','five',
    'six','seven','eight','nine',
}

def is_phonetic_sequence(span: str, threshold: float = 0.5) -> bool:
    """
    span 50%  NATO   True.
    'Oscar Kilo Alpha Golf Lima' → True (  callsign)
    """
    words = span.lower().split()
    if not words:
        return False
    phonetic_cnt = sum(1 for w in words if w in NATO_PHONETIC)
    return phonetic_cnt / len(words) >= threshold

def should_correct(raw_span: str, slot_db,
                   keep_thr: float = 0.25,
                   check_phonetic: bool = False) -> bool:
    """
       .
    Returns False ( ) :
      1. NATO     (callsign )
      2.  DB     (Lev/len < keep_thr)
    Returns True  →
    """
    if not raw_span:
        return False

    if check_phonetic and is_phonetic_sequence(raw_span):
        # [Korean print removed]
        return False

    candidates = slot_db.search(raw_span, k=5)
    if not candidates:
        return True
    best_dist = min(
        editdistance.eval(raw_span.lower(), c.split('|')[0].strip().lower())
        for c in candidates
    )
    ratio = best_dist / max(len(raw_span), 1)
    if ratio < keep_thr:
        # [Korean print removed]')
        return False

    return True

def _call_llm_slot_select_v3(asr_text: str, cands: dict,
                              comm_info: dict, fk_facility: str,
                              quality: dict = None) -> dict:
    cand_lines = '\n'.join(f'  {s}: {v}' for s, v in cands.items())

    _cmd     = comm_info.get('detected_cmd', 'unknown')
    _ctype   = comm_info.get('comm_type',    'unknown')
    _struct  = comm_info.get('structure',    '')
    _example = comm_info.get('example',      '')

    _fac_hint = (
        f'\nNote: Recording metadata suggests facility = "{fk_facility}". '
        f'Use only if FACILITY candidates support it.'
        if fk_facility and 'FACILITY' in cands
        else ''  # ← FACILITY hint
    )

    # -- Build quality section for prompt --
    _q = quality or {}
    _level = _q.get('correction_level', 'LIGHT')
    _correction_guide = {
        'SKIP':      'ASR appears highly accurate. Make minimal changes only.',
        'LIGHT':     'Apply light correction: fix obvious phonetic errors only.',
        'AGGRESSIVE':'Apply thorough correction: ASR has significant errors.',
        'REJECT':    'ASR output too degraded. Return null for all slots.',
    }
    _quality_section = f"""
## ASR Quality Assessment (reference-free, ICAO-based)
- Model confidence  : {_q.get('confidence', 'N/A')}
- ICAO vocab coverage: {_q.get('vocab_coverage', 'N/A')}
- Entity format score: {_q.get('entity_score', 'N/A')}
- Overall quality   : {_q.get('overall', 'N/A')}  -> Correction level: {_level}
- Instruction       : {_correction_guide.get(_level, '')}
""" if quality else ""

    prompt = f"""You are an expert Air Traffic Control (ATC) transcript corrector with deep knowledge of ICAO standards.

## ASR Output
"{asr_text}"

{_quality_section}
## Detected Communication Type
Command : {_cmd} | Type: {_ctype}
Expected structure: {_struct}
ICAO example: {_example}{_fac_hint}

## Slot Candidates
Retrieved from ATC domain knowledge base via semantic search:
{cand_lines}

## ICAO Entity Definitions
Apply these definitions strictly when selecting candidates:

FACILITY
  - An ATC unit name consisting of airport name + service type
  - Service types: Tower, Ground, Approach, Radar, Control, Departure, Delivery
  - Valid: "Sydney Tower", "Zurich Approach", "Prague Radar"
  - Select only if the ASR contains words that sound like a place name or service type
  - CRITICAL: NEVER assign QNH values, frequencies, or wind information to FACILITY

CALLSIGN
  - An aircraft identifier: ICAO airline telephony designator + flight number
  - Telephony designators are spoken names (e.g. "Swiss", "Speedbird", "Lufthansa")
  - Flight numbers are spoken digit-by-digit (e.g. "Two Zero Four", not "204")
  - May include suffix: "Heavy", "Super"
  - Invalid: pure measurements, distances, directions, weather values

INSTRUCTION
  - A standard ICAO phraseology command issued by controller or pilot
  - Must be a recognised ATC phrase (climb, descend, cleared, contact, squawk, etc.)
  - Select the phrase that best matches the phonetic content of the ASR
  - CRITICAL: INSTRUCTION slot = command verb ONLY. Do NOT include VALUE in this slot.
    e.g. "climb" is correct. "climb to flight level one hundred" is WRONG for this slot.
  - PILOT speech rule: if ASR contains (we are, not ready, request, wilco, roger, negative),
    this is PILOT speech. Do NOT replace with an ATC command (e.g. "line up and wait").
    Return null for INSTRUCTION if the original is a pilot state report.

RUNWAY
  - Format: "runway" + two-digit number (01–36) + optional designator (L/R/C)
  - Valid: "runway 28L", "runway 09", "runway 36R"
  - Invalid: headings, bearings, wind directions, altitudes, frequencies

QNH
  - Altimeter setting value, preceded by "QNH" keyword in ASR
  - Format: four spoken digits (e.g. "one zero two three")
  - MUST NOT be placed in FACILITY slot

WIND
  - Wind information, preceded by "wind" keyword in ASR
  - Includes direction in degrees and speed in knots, or "calm"
  - Example: "two seven zero degrees one zero knots", "calm"
  - MUST NOT be placed in FACILITY or CALLSIGN slot

FREQUENCY
  - Radio frequency, contains "decimal" keyword
  - Example: "one two three decimal two five five"
  - MUST NOT be placed in FACILITY slot

## Correction Principles
1. Phonetic mapping: ASR errors often arise from similar-sounding words
   (e.g. "come"→"climb", "Cindy"→"Sydney", "ready"→"radar", "fore"→"four")
2. Context consistency: chosen slots must form a coherent ATC utterance
3. Confidence: if uncertain between candidates, prefer the one with higher
   phonetic similarity to the ASR token
4. Absence: if no candidate plausibly matches any token in the ASR, return null
5. Speaker protection: pilot state reports must NOT be converted to ATC commands.
   Pilot phrases (we are not ready, request, negative, wilco) express state, not instructions.
6. No hallucination: NEVER select a value that has zero phonetic overlap with ASR tokens.

## Output
Return ONLY a valid JSON object. No explanation, no markdown fences.
{{"FACILITY": "...", "CALLSIGN": "...", "INSTRUCTION": "...", "RUNWAY": "...", "QNH": "...", "WIND": "...", "FREQUENCY": "...", "GREETING": "..."}}"""

    raw = call_claude(prompt, max_tokens=300).replace('```json','').replace('```','').strip()
    try:
        return _json.loads(raw)
    except _json.JSONDecodeError:
        # [Korean print removed]
        return {}

def reconstruct_v3(slots_corr: dict, slots_raw: dict,
                   asr_text: str, cands: dict) -> str:
    txt_lower = asr_text.lower()
    parts     = []
    order     = ['FACILITY', 'GREETING', 'CALLSIGN', 'INSTRUCTION', 'QNH', 'WIND', 'RUNWAY', 'FREQUENCY']

    for key in order:
        val = slots_corr.get(key)
        if not val:
            continue

        if key == 'FACILITY':
            if 'FACILITY' not in cands:
                continue
            loc_kws = ['tower','ground','approach','radar','control',
                       'centre','center','departure','arrival']
            if not any(kw in txt_lower for kw in loc_kws):
                continue
            if slots_raw.get('FACILITY') is None:
                continue

        if key == 'INSTRUCTION':
            val_words = val.lower().split()
            if len(val_words) > 4:
                icao_cmds = ['climb', 'descend', 'cleared', 'contact', 'squawk',
                             'maintain', 'turn', 'hold', 'taxi', 'report',
                             'line up', 'cross', 'go around', 'continue']
                matched_cmd = next(
                    (cmd for cmd in icao_cmds if cmd in val.lower()), None
                )
                if matched_cmd:
                    val = matched_cmd

        parts.append(val)

    parts.extend(slots_corr.get('EXTRA', []))
    return ' '.join(parts)

def slot_filling_llm_correct_v3(asr_text: str, file_key: str = '') -> dict:
    # Step 1: Slot mapping
    slots_raw = map_to_slots(asr_text)

    _fk_facility = ''
    if file_key and 'MHz' in file_key and file_key.count('_') >= 3:
        _meta       = parse_file_key(file_key)
        _fk_fac_raw = FACILITY_DB.get(_meta['icao'], {}).get(
            'services', {}).get(_meta['service'], '')
        _fac_kws = ['tower','ground','approach','radar','control',
                    'departure','arrival','centre','center']
        if _fk_fac_raw and any(
                editdistance.eval(w, kw) / len(kw) <= 0.40
                for w in asr_text.lower().split()
                for kw in _fac_kws):
            _fk_facility = _fk_fac_raw

    comm_info = detect_comm_structure(asr_text)
    # [Korean print removed]

    cands = _collect_candidates_v3(slots_raw)
    if not cands:
        return slot_filling_correct(asr_text, file_key)
    # [Korean print removed]

    # Pass quality assessment to LLM
    _quality = wav_quality.get(file_key, None) if 'wav_quality' in dir() else None

    # REJECT level: skip LLM, return ASR as-is
    if _quality and _quality['correction_level'] == 'REJECT':
        print(f'  [REJECT] Quality too low ({_quality["overall"]}) - returning original ASR')
        return {
            'asr_output': asr_text, 'rag_output': asr_text,
            'slots_raw': {}, 'slots_corr': {}, 'corrections': [],
            'quality': _quality,
        }

    # SKIP level: skip LLM, use Levenshtein only
    if _quality and _quality['correction_level'] == 'SKIP':
        print(f'  [SKIP] High quality ({_quality["overall"]}) - Levenshtein only')
        lev_slots   = rerank_slots(slots_raw)
        lev_slots['EXTRA'] = slots_raw.get('EXTRA', [])
        return {
            'asr_output': asr_text, 'rag_output': reconstruct_v3(lev_slots, slots_raw, asr_text, cands),
            'slots_raw': slots_raw, 'slots_corr': lev_slots, 'corrections': [],
            'quality': _quality,
        }

    llm_result = _call_llm_slot_select_v3(asr_text, cands, comm_info, _fk_facility, _quality)

    lev_slots   = rerank_slots(slots_raw)
    final_slots = dict(lev_slots)
    for k in ['FACILITY','CALLSIGN','INSTRUCTION','RUNWAY',
              'QNH','WIND','FREQUENCY','GREETING']:
        llm_val = llm_result.get(k)
        if llm_val and str(llm_val).lower() not in ('null','none',''):
            final_slots[k] = llm_val
        elif slots_raw.get(k):
    final_slots['EXTRA'] = slots_raw.get('EXTRA', [])

    _all_empty_v3 = all(
        not final_slots.get(k)
        for k in ['FACILITY','CALLSIGN','INSTRUCTION','RUNWAY',
                  'QNH','WIND','FREQUENCY','GREETING']
    ) and not final_slots.get('EXTRA')
    rag_output = asr_text if _all_empty_v3 else reconstruct_v3(final_slots, slots_raw, asr_text, cands)

    log = []
    for k in ['FACILITY','CALLSIGN','INSTRUCTION','RUNWAY','QNH','WIND','FREQUENCY','GREETING']:
        raw_v = slots_raw.get(k)
        fin_v = final_slots.get(k)
        if raw_v and fin_v and str(raw_v).lower() != str(fin_v).lower():
            src = 'LLM' if llm_result.get(k) else 'Levenshtein'
            log.append({'slot': k, 'before': raw_v, 'after': fin_v, 'source': src})

    return {
        'asr_output':  asr_text,
        'slots_raw':   slots_raw,
        'slots_corr':  final_slots,
        'rag_output':  rag_output,
        'corrections': log,
        'comm_info':   comm_info,
    }

print('Slot Filling + LLM Re-ranking v3\n')
wav_llm_results = {}

if not wav_asr_results:
    print("[WARN] wav_asr_results empty. Run cell 5-1 first.")
else:
    for fkey, asr_txt in wav_asr_results.items():
        print(f'\n{"="*62}')
        print(f'File: {fkey}')
        print(f'ASR : {asr_txt[:90]}')
        try:
            result = slot_filling_llm_correct_v3(asr_txt, fkey)
            wav_llm_results[fkey] = result
            print(f'RAG : {result["rag_output"][:90]}')
            if result['corrections']:
                for c in result['corrections']:
                    print(f'  [{c["slot"]}|{c.get("source","?")}] '
                          f'{c["before"]} → {c["after"]}')
            else:
                print('  (no correction)')
        except Exception as _ex:
            import traceback
            print(f'  [ERR] {type(_ex).__name__}: {_ex}')
            traceback.print_exc()
            wav_llm_results[fkey] = {
                'asr_output': asr_txt, 'rag_output': asr_txt,
                'slots_raw': {}, 'slots_corr': {}, 'corrections': []
            }

    print(f'\nDone: {len(wav_llm_results)} file(s)')

    if wav_all_results and wav_llm_results:
        print(f'\n{"─"*100}')
        print(f'{"File":28s} | {"ASR":28s} | {"Levenshtein":28s} | {"LLM v3":28s}')
        print('─' * 100)
        for fk in wav_llm_results:
            asr_s = wav_asr_results.get(fk, '')[:26]
            lev_s = wav_all_results.get(fk, {}).get('rag_output', '')[:26]
            llm_s = wav_llm_results[fk].get('rag_output', '')[:26]
            print(f'{fk[-28:]:28s} | {asr_s:28s} | {lev_s:28s} | {llm_s:28s}')

Slot Filling + LLM Re-ranking v3


File: YSSY_SYDNEY_Tower_120_5MHz_20210501_200237
ASR : toy waiting on release from
  NER raw: [('NAMED', 'toy', 0.99), ('COMMAND', 'waiting on release from', 0.98)]
RAG : line up and wait
  [INSTRUCTION|LLM] waiting on release from → line up and wait

File: YSSY_SYDNEY_Tower_120_5MHz_20210502_004327
ASR : six contact part contact departure in three two six
  NER raw: [('VALUE', 'six', 0.98), ('COMMAND', 'contact part', 0.87), ('COMMAND', 'contact', 0.92), ('COMMAND', 'departure', 0.47), ('VALUE', 'in three two six', 0.89)]
RAG : Sydney Tower Six contact Departure in three two six six in three two six
  [FACILITY|LLM] in three two six → Sydney Tower
  [INSTRUCTION|LLM] contact part → contact

File: YSSY_SYDNEY_Tower_120_5MHz_20210502_012007
ASR : runway three four left cleared to land and request or three two bravo four available jet f
  NER raw: [('VALUE', 'runway three four left', 1.0), ('COMMAND', 'cleared to land', 1.0), ('COMMAND', 'request', 0.99